# Resolving higher stages and comparing bases fairly

These experiments follow the previously reviewed limitations. They distinguish **a physical field basis**, **a circuit realization of that basis**, and **a fitted terminal transfer**. New results have Codex self-review; independent Claude review is pending. All listed comparisons use the stated actual excitation.

## Three-stage round-wire CLN

For the zero-shift round-wire impedance, let $u=s\mu\sigma r^2$ and normalize by $R_b=1/(\pi\sigma r^2 h)$. The response is $\sqrt{u}I_0(\sqrt{u})/[2I_1(\sqrt{u})]$. Mathematica ([WLS source](../mathematica/derive_round_wire_three_stages.wls)) and independent SymPy rational power-series extraction give $R/R_b=(1,3,5,7)$ and $L/(R_b\mu\sigma r^2)=(1/8,1/16,1/24)$. The underlying modified-Bessel continued fraction is given in [NIST DLMF 10.33.1](https://dlmf.nist.gov/10.33.E1).

Boundary conditions, load gate and ICCG shift are unchanged. Extra quadrature and FE order are independent controls. The sweep tests three stages, eight discretization settings and all three formulations. Passing a solve alone is not an accuracy result. These are conductor-interior coefficients for the prescribed initial axial field; terminal coefficients use the length normalization explained in notebook 06.

In [1]:
import json,numpy as np
from pathlib import Path
from IPython.display import display,Markdown
DATA=Path('docs/data') if Path('docs/data').exists() else Path('data')
def read(name):return json.loads((DATA/(name+'.json')).read_text(encoding='utf-8'))
h=read('higher_stages')
rows=['| Formulation | FE order | h [mm] | max R error [%] | max L error [%] |','|---|---:|---:|---:|---:|']
for v in h['cases']:
    if v['bonus_intorder']==8 and v['order']>=3:
        rows.append(f"| {v['formulation']} | {v['order']} | {1000*v['maxh']:.1f} | {100*max(v['R_relative_error']):.6f} | {100*max(v['L_relative_error']):.6f} |")
display(Markdown('\n'.join(rows)))
print('Recorded gate/solver failures:',[(v['formulation'],v['order'],v['bonus_intorder'],v['error']) for v in h['cases'] if v['status']!='pass'])


| Formulation | FE order | h [mm] | max R error [%] | max L error [%] |
|---|---:|---:|---:|---:|
| CLN_APhi | 3 | 3.0 | 3.165748 | 0.628296 |
| CLN_T_Omega | 3 | 3.0 | 4.189945 | 0.701670 |
| CLN_AT | 3 | 3.0 | 0.907121 | 0.081427 |
| CLN_APhi | 4 | 3.0 | 0.061903 | 0.005523 |
| CLN_T_Omega | 4 | 3.0 | 0.081423 | 0.005586 |
| CLN_AT | 4 | 3.0 | 0.020449 | 0.000098 |
| CLN_APhi | 3 | 1.5 | 0.077296 | 0.011438 |
| CLN_T_Omega | 3 | 1.5 | 0.090125 | 0.012812 |
| CLN_AT | 3 | 1.5 | 0.013438 | 0.001427 |

Recorded gate/solver failures: [('CLN_T_Omega', 1, 4, 'Incompatible load [T stage 3]: kernel fraction 1.224e-08 exceeds 1.000e-08')]


The coarse first-order T–Omega third-stage load fails with extra quadrature 4 and passes with 8. Its element errors remain large. Increasing FE order improves higher-stage accuracy much more than increasing quadrature once the load is compatible. At order 3 and h=1.5 mm, all seven elements of all three formulations are within 0.2% of the independently extracted coefficients (observed maximum about 0.0902%). Both h-refinement and p-refinement reduce the last-stage errors. This resolves the tested three-stage round-wire case, not arbitrary stage counts, nonzero shifts, nonlinear 3D fields or multiply connected domains.

## Same training grid and state count: terminal Foster versus field models

Proper Foster fits have the same number of finite poles and no ideal-inductor tail or infinite pole. They use the same 31 frequency samples as the common-excitation POD experiment, with multipoint-CLN seeds. These are matched state counts and training grids, **not matched offline budgets**: CLN uses its stated real shifts; POD receives full fields; Foster receives terminal samples plus its seed and optimizer. DC-constrained fits fix both the known zero DC value and normalized unit static slope. Free-DC fits are warm-started from the constrained fit; a nonzero offset can dominate low-frequency relative error even if the slope error is small. Fit timings exclude full reference and seed construction.

The high-band hold-outs exclude training points and measure interpolation; the low-band errors below 100 Hz measure extrapolation. Every model is compared against the same *discrete* internal-conductor response. A terminal fit does not provide field or Joule-loss reconstruction. Static-air or DC-resistance dilution is not used in this reaction-transfer metric.

In [2]:
d=read('information_fine')
rows=['| Model | States | high-band hold-out max [%] | static slope error [%] | low-band max [%] |','|---|---:|---:|---:|---:|']
for name,v in d['runs'].items():
    rows.append(f"| {name} | {v['rank']} | {100*v['holdout_high_band_max']:.7f} | {100*v['DC_slope_error']:.7f} | {100*v['low_band_max']:.7f} |")
display(Markdown('\n'.join(rows)))


| Model | States | high-band hold-out max [%] | static slope error [%] | low-band max [%] |
|---|---:|---:|---:|---:|
| ordinary CLN 4 | 4 | 1.0580336 | 0.0000130 | 0.0000130 |
| multipoint CLN 4 | 4 | 0.0735952 | 0.0000477 | 0.0000482 |
| field POD 4 | 4 | 0.0303762 | 0.0153918 | 0.0153926 |
| bulk2 + boundary-response POD 4 | 4 | 0.0491878 | 0.0032662 | 0.0032665 |
| Foster DC constrained 4 | 4 | 0.0350981 | 0.0000000 | 0.0003300 |
| Foster DC free 4 | 4 | 0.0341334 | 0.0120400 | 106.7296598 |
| ordinary CLN 8 | 8 | 0.0408679 | 0.0000000 | 0.0000000 |
| multipoint CLN 8 | 8 | 0.0003214 | 0.0000000 | 0.0000000 |
| field POD 8 | 8 | 0.0000585 | 0.0000236 | 0.0000236 |
| bulk2 + boundary-response POD 8 | 8 | 0.0000465 | 0.0000267 | 0.0000267 |
| Foster DC constrained 8 | 8 | 0.0000413 | 0.0000000 | 0.0000004 |
| Foster DC free 8 | 8 | 0.0000403 | 0.0000170 | 0.1326945 |

At four states on the fine mesh, full-field POD has smaller terminal error than proper Foster, while Foster is more accurate than single/multipoint CLN in the band. At eight states, proper Foster has the smallest terminal error in this run. Freeing DC improves its in-band error only modestly, while the four-state model has over 100% relative error at the low-band end because of its nonzero DC offset. The eight-state free-DC model also worsens low-frequency extrapolation. These are local optimizer results, not a global optimum or a universal ordering; sacrificing DC is not necessary for good high-frequency reach here. At eight states every band-aware method already has very small discrete terminal error. Mesh error remains a separate uncertainty, so none of these sub-percent reduction errors alone proves physical high-frequency accuracy.

For a **fixed reduced field space**, diagonalizing its positive energy pencil gives the same terminal response. Foster versus Cauer accuracy is then identical; an accuracy difference requires different truncation, fitting, basis or numerical error. Surface enrichment does not change that identity.

## Nonlinear held-out amplitudes and matched retained rank

The nonlinear 2D test uses the exact full discrete exterior surface operator, the same actual physical excitation and the same time-discrete constitutive law. POD is trained on the full nonlinear trajectory at amplitude 1.0. Amplitudes 0.5 and 1.5 are held out, with unchanged waveform and frequency. The results do not test new frequencies or new material laws.

The surface construction preserves two bulk CLN directions and selects energy-weighted residual directions from auxiliary Steklov-response blocks. It receives additional auxiliary loads; it is not a basis generated solely by the physical excitation. This SVD compresses auxiliary responses and is POD-like, but it is not POD of actual nonlinear field trajectories. The tested selection is one specific enrichment design.

Only requested ranks that survive rank revelation are compared. A twice-reorthogonalized actual-source Krylov span is a numerical control; its Galerkin model is not automatically a derived physical Cauer ladder.

In [3]:
nonlinear=read('nonlinear_matched_fine')
rows=['| Model | amplitude | states | conductor linkage error [%] | Joule error [%] | field energy RMS [%] |','|---|---:|---:|---:|---:|---:|']
for name,v in nonlinear['runs'].items():
    if name.endswith('amplitude 1.0'):continue
    model,amp=name.split(' amplitude ')
    rows.append(f"| {model} | {amp} | {v['rank']} | {100*v['rod_linkage_relative_error']:.6f} | {100*v['joule_relative_error']:.6f} | {100*v['field_energy_rms']:.6f} |")
display(Markdown('\n'.join(rows)))
print('Requested ranks not retained:',[v for v in nonlinear['basis_audit'] if v['rank']!=v['requested_rank']])
print('Same-space modal checks:',nonlinear['same_basis_modal_checks'])


| Model | amplitude | states | conductor linkage error [%] | Joule error [%] | field energy RMS [%] |
|---|---:|---:|---:|---:|---:|
| single CLN 9 | 0.5 | 9 | 0.037385 | 0.232766 | 2.063289 |
| stabilized actual-source Krylov 9 | 0.5 | 9 | 0.037385 | 0.232766 | 2.063289 |
| multipoint actual source 9 | 0.5 | 9 | 0.037305 | 0.232455 | 2.064086 |
| protected bulk2 + auxiliary surface 9 | 0.5 | 9 | 0.219086 | 1.094937 | 4.785461 |
| nonlinear training POD 9 | 0.5 | 9 | 0.002238 | 0.020106 | 0.375191 |
| stabilized actual-source Krylov 19 | 0.5 | 19 | 0.034299 | 0.193680 | 1.916112 |
| protected bulk2 + auxiliary surface 19 | 0.5 | 19 | 0.181333 | 0.948658 | 4.473027 |
| nonlinear training POD 19 | 0.5 | 19 | 0.000150 | 0.000893 | 0.090395 |
| single CLN 9 | 1.5 | 9 | 0.042863 | 0.204939 | 0.987404 |
| stabilized actual-source Krylov 9 | 1.5 | 9 | 0.042863 | 0.204939 | 0.987404 |
| multipoint actual source 9 | 1.5 | 9 | 0.042775 | 0.204363 | 0.987360 |
| protected bulk2 + auxiliary surface 9 | 1.5 | 9 | 0.242408 | 0.946263 | 2.159255 |
| nonlinear training POD 9 | 1.5 | 9 | 0.002019 | 0.017458 | 0.192996 |
| stabilized actual-source Krylov 19 | 1.5 | 19 | 0.039763 | 0.168569 | 0.909770 |
| protected bulk2 + auxiliary surface 19 | 1.5 | 19 | 0.198385 | 0.821467 | 2.016295 |
| nonlinear training POD 19 | 1.5 | 19 | 0.000158 | 0.001372 | 0.048807 |

Requested ranks not retained: [{'label': 'single CLN 19', 'requested_rank': 19, 'rank': 18}, {'label': 'multipoint actual source 19', 'requested_rank': 19, 'rank': 9}]
Same-space modal checks: {'1.0': {'trace_relative_difference': 1.277364976550993e-10, 'joule_relative_difference': 6.605405111770324e-10}, '0.5': {'trace_relative_difference': 3.443748807173472e-10, 'joule_relative_difference': 2.6595237123672177e-09}, '1.5': {'trace_relative_difference': 1.1506257015964724e-10, 'joule_relative_difference': 7.055691586543844e-10}}


The actual nonlinear-trajectory POD has the smallest linkage, Joule and energy-norm field errors in these tested held-out amplitudes. The tested protected two-bulk-mode surface enrichment is worse than the same-rank actual-source linear spaces. Therefore **adding surface modes alone does not guarantee improvement**; selection, weighting and preservation of the bulk response matter. This is a counterexample to unconditional improvement, not evidence that every surface enrichment is poor. It does not replace the previous higher-rank enrichment experiment with a claim of universal CLN or POD superiority.

Diagonalizing the same enriched linear pencil while retaining its nonlinear controlled sources reproduces the coupled nonlinear model to the stated solver accuracy. Those sources remain coupled after changing coordinates. It is a realizable modal representation, not independent linear Foster branches describing a nonlinear material.

## Reproduce

```text
wolframscript -file mathematica/derive_round_wire_three_stages.wls docs/data/round_wire_three_stages.json
python validation/validate_higher_stages.py --output docs/data/higher_stages.json
python validation/surface_hybrid/compare_information_budget.py --maxh .004 --output docs/data/information_coarse.json
python validation/surface_hybrid/compare_information_budget.py --maxh .003 --output docs/data/information_fine.json
python validation/surface_hybrid/compare_nonlinear_bases.py --h-over-a .3 --output docs/data/nonlinear_matched_coarse.json
python validation/surface_hybrid/compare_nonlinear_bases.py --h-over-a .2 --output docs/data/nonlinear_matched_fine.json
python validation/validate_research.py
```

Source identities and available failed-rank records are retained. The scopes above make definite statements about tested cases; they do not turn a finite campaign into a universal ranking.

## Full-model mesh uncertainty, independently solved

A sparse mixed A–phi solve bypasses the dense eigensystem reduction. It uses the same geometry, source and boundary conditions and verifies the original equations. Rescaling the mixed unknown as $\psi=s\phi$ gives the symmetric block system $[K+sM,C^T;C,K_\phi/s]$; diagonal scaling improves its numerical residual. Penalty-weight invariance is tested at endpoint frequencies. The h=3 mm response and static normalization agree independently with the stored modal reference.

The next table separates normalized reaction error from physical susceptibility error, which includes the changing static normalization. Mesh-to-mesh differences are estimates, not certified continuum error bounds. Four representative frequencies are sampled; no unsampled-band bound is claimed.

In [4]:
mesh=read('mesh_reference')
rows=['| Mesh pair [mm] | static difference [%] | normalized response max [%] | physical susceptibility max [%] |','|---|---:|---:|---:|']
for a,b in zip(mesh['meshes'][:-1],mesh['meshes'][1:]):
    za=np.asarray(a['reference_real'])+1j*np.asarray(a['reference_imag']);zb=np.asarray(b['reference_real'])+1j*np.asarray(b['reference_imag'])
    rows.append(f"| {1000*a['maxh']:.1f} / {1000*b['maxh']:.1f} | {100*abs(a['static_H']/b['static_H']-1):.4f} | {100*max(abs(za/zb-1)):.4f} | {100*max(abs(a['static_H']*za/(b['static_H']*zb)-1)):.4f} |")
display(Markdown('\n'.join(rows)))
print('Maximum original-equation residual:',max(v['original_residual_max'] for v in mesh['meshes']))


| Mesh pair [mm] | static difference [%] | normalized response max [%] | physical susceptibility max [%] |
|---|---:|---:|---:|
| 3.0 / 2.0 | 1.3330 | 0.6030 | 1.5746 |
| 2.0 / 1.5 | 1.5324 | 0.7213 | 2.1154 |

Maximum original-equation residual: 2.5631520641478466e-13


The last pair still differs by about 2.1% in physical susceptibility at 1 MHz, and the differences are not yet decreasing monotonically. This mesh sequence does **not** establish 1% continuum accuracy. It does establish that mesh uncertainty exceeds the small reduction errors in these examples. A device-level Foster/CLN ranking therefore cannot be inferred from these discrete terminal errors alone.

Reproduce this full-model check with:
```text
python validation/surface_hybrid/mesh_reference.py --output docs/data/mesh_reference.json
```
